# NB02: Transform and Clean the SEC Data

## 1. Import libraries and define project information

In this notebook, I transform the raw JSON files collected in NB01 into a clean dataset for analysis.

I import the required Python libraries, define the location of the raw data folder, and create a dictionary containing the companies included in the project. This information will be used throughout the notebook to load, organise, and compare financial data across sectors.

In [15]:
import json
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

companies = {
    "MSFT": {"name": "Microsoft", "sector": "Technology"},
    "NVDA": {"name": "Nvidia", "sector": "Technology"},
    "AAPL": {"name": "Apple", "sector": "Technology"},
    "PFE": {"name": "Pfizer", "sector": "Healthcare"},
    "JNJ": {"name": "Johnson & Johnson", "sector": "Healthcare"},
    "SYK": {"name": "Stryker", "sector": "Healthcare"},
    "XOM": {"name": "ExxonMobil", "sector": "Energy"},
    "CVX": {"name": "Chevron", "sector": "Energy"},
    "DUK": {"name": "Duke Energy", "sector": "Energy"},
}

## 1. Load and combine the raw SEC data

The raw financial data collected in NB01 is stored as separate JSON files, with one file for each company and financial concept.

In this step, I read every JSON file from the `data/raw` folder and extract the relevant financial observations. For each observation, I keep the company ticker, financial concept (tag), reporting period, reported value (revenue), and filing date.

In [16]:
records = []

for path in sorted(RAW_DIR.glob("*.json")):
    ticker, tag = path.stem.split("_", 1)

    with open(path) as f:
        data = json.load(f)

    for obs in data["units"]["USD"]:
        if "start" not in obs:
            continue
        records.append({
            "ticker": ticker,
            "tag": tag,
            "start": obs["start"],
            "end": obs["end"],
            "val": obs["val"],
            "filed": obs["filed"],
        })

long = pd.DataFrame(records)
print(len(long), "observations read")
long.sample(10)

8428 observations read


,ticker,tag,start,end,val,filed
216,AAPL,NetIncomeLoss,2012-04-01,2012-06-30,8824000000,2012-07-25
4426,MSFT,ResearchAndDevelopmentExpense,2015-07-01,2015-12-31,5862000000,2017-01-26
2005,DUK,NetCashProvidedByUsedInOperatingActivities,2024-01-01,2024-12-31,12328000000,2026-02-26
585,AAPL,ResearchAndDevelopmentExpense,2008-09-28,2009-09-26,1333000000,2010-10-27
254,AAPL,NetIncomeLoss,2013-06-30,2013-09-28,7512000000,2015-01-28
7454,SYK,ResearchAndDevelopmentExpense,2019-01-01,2019-09-30,717000000,2020-10-30
6320,PFE,ResearchAndDevelopmentExpense,2011-07-04,2011-10-02,2188000000,2011-11-10
6373,PFE,ResearchAndDevelopmentExpense,2015-06-29,2015-09-27,1722000000,2016-11-10
1855,CVX,Revenues,2025-07-01,2025-09-30,49726000000,2025-11-06
419,AAPL,NetIncomeLoss,2021-09-26,2022-03-26,59640000000,2023-05-05


## 3. Clean the reporting periods

The SEC data contains observations covering different reporting periods and, some have multiple filings for the same financial year.

To create a consistent annual dataset, I first calculated the length of each reporting period and kept only observations that covered about one year. Then I assigned each observation to the calendar year that contained most of its reporting period, because some companies have financial years that end in early January. Finally, when a company reported the same financial concept for the same year more than once, I kept the most recently filed value.

These steps ensure that the final dataset contains one annual observation for each company and financial concept.

In [17]:
long["days"] = (pd.to_datetime(long["end"]) - pd.to_datetime(long["start"])).dt.days
long = long[long["days"].between(350, 380)]

# Assign each observation to the calendar year it covers the most using the midpoint date.
start = pd.to_datetime(long["start"])
end = pd.to_datetime(long["end"])
long["year"] = (start + (end - start) / 2).dt.year

long = long.sort_values("filed").groupby(["ticker", "tag", "year"], as_index=False).last()

print(len(long), "annual observations")

784 annual observations


## 4. Check for overlapping revenue tags

Companies may report revenue using different concepts. Before selecting a single revenue measure, I check whether multiple revenue tags exist for the same company and year.

Identifying these overlaps helps avoid mixing different revenue definitions and ensures that a consistent revenue measure is used for the final analysis.

In [18]:
revenue_tags = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "RevenueFromContractWithCustomerIncludingAssessedTax",
    "Revenues",
]

rev = long[long["tag"].isin(revenue_tags)]
# Group the data by company and year, then keep only groups with multiple revenue tags.
overlap = rev.groupby(["ticker", "year"]).filter(lambda g: len(g) > 1)

overlap[["ticker", "year", "tag", "val"]].head(20)

,ticker,year,tag,val
69,AAPL,2017,RevenueFromContractWithCustomerExcludingAssess...,229234000000
70,AAPL,2018,RevenueFromContractWithCustomerExcludingAssess...,265595000000
79,AAPL,2017,Revenues,229234000000
80,AAPL,2018,Revenues,265595000000
156,CVX,2018,RevenueFromContractWithCustomerExcludingAssess...,158902000000
157,CVX,2019,RevenueFromContractWithCustomerExcludingAssess...,139865000000
158,CVX,2020,RevenueFromContractWithCustomerExcludingAssess...,94471000000
159,CVX,2021,RevenueFromContractWithCustomerExcludingAssess...,155606000000
160,CVX,2022,RevenueFromContractWithCustomerExcludingAssess...,235717000000
161,CVX,2023,RevenueFromContractWithCustomerExcludingAssess...,196913000000


## 5. Select consistent revenue definitions and create the analysis panel

Some companies report revenue using different SEC XBRL tags.

To ensure a consistent measure of revenue over time, I select one revenue tag for each company and use it throughout the analysis. The selected revenue tags are then renamed to a common "Revenue" category.

Finally, the data is reshaped into a panel format where each row represents one company in one year and each column represents a financial concept.

In [19]:
revenue_choice = {
    "MSFT": "RevenueFromContractWithCustomerExcludingAssessedTax",
    "AAPL": "RevenueFromContractWithCustomerExcludingAssessedTax",
    "JNJ":  "RevenueFromContractWithCustomerExcludingAssessedTax",
    "DUK":  "RevenueFromContractWithCustomerIncludingAssessedTax",
    "NVDA": "Revenues",
    "SYK":  "Revenues",
    "XOM":  "Revenues",
    "CVX":  "Revenues",
    "PFE":  "Revenues",  
}

# Identify revenue rows and check whether they match the chosen revenue tag.
is_revenue = long["tag"].isin(revenue_tags)
chosen = long["ticker"].map(revenue_choice) == long["tag"]
# Keep only the selected revenue tag for each company.
tidy = long[~is_revenue | chosen].copy()

# Rename all selected revenue tags to a common name.
tidy.loc[tidy["tag"].isin(revenue_tags), "tag"] = "Revenue"

panel = tidy.pivot(index=["ticker", "year"], columns="tag", values="val").reset_index()
panel.columns.name = None

panel["sector"] = panel["ticker"].map(lambda t: companies[t]["sector"])
panel["name"] = panel["ticker"].map(lambda t: companies[t]["name"])

panel.head(10)

,ticker,year,NetCashProvidedByUsedInOperatingActivities,NetIncomeLoss,PaymentsToAcquirePropertyPlantAndEquipment,ResearchAndDevelopmentExpense,Revenue,sector,name
0,AAPL,2007,5.470000e+09,3.495000e+09,NaN,7.820000e+08,NaN,Technology,Apple
1,AAPL,2008,9.596000e+09,6.119000e+09,NaN,1.109000e+09,NaN,Technology,Apple
2,AAPL,2009,1.015900e+10,8.235000e+09,NaN,1.333000e+09,NaN,Technology,Apple
3,AAPL,2010,1.859500e+10,1.401300e+10,NaN,1.782000e+09,NaN,Technology,Apple
4,AAPL,2011,3.752900e+10,2.592200e+10,NaN,2.429000e+09,NaN,Technology,Apple
5,AAPL,2012,5.085600e+10,4.173300e+10,NaN,3.381000e+09,NaN,Technology,Apple
6,AAPL,2013,5.366600e+10,3.703700e+10,8.165000e+09,4.475000e+09,NaN,Technology,Apple
7,AAPL,2014,NaN,3.951000e+10,9.571000e+09,6.041000e+09,NaN,Technology,Apple
8,AAPL,2015,8.126600e+10,5.339400e+10,1.124700e+10,8.067000e+09,NaN,Technology,Apple
9,AAPL,2016,6.623100e+10,4.568700e+10,1.273400e+10,1.004500e+10,NaN,Technology,Apple


## 6. Restrict the analysis period and check data availability

To ensure comparability across companies, the analysis is restricted to the 2018–2024 period.

Before calculating financial ratios, I check how many companies have available values for each financial concept in each year. This helps identify missing data and ensures that sector comparisons are based on sufficient observations.

findings: 
All 9 companies reported Operating Cash Flow,
All 9 companies reported Net Income
,Only 8 companies reported Capex
,Only 8 companies reported R&D
,Only 7 companies reported Revenue

In [20]:
panel = panel[panel["year"].between(2018, 2024)]

# how many companies have a value for each column, by year?
panel.set_index(["year", "ticker"]).notna().groupby("year").sum()

,NetCashProvidedByUsedInOperatingActivities,NetIncomeLoss,PaymentsToAcquirePropertyPlantAndEquipment,ResearchAndDevelopmentExpense,Revenue,sector,name
year,,,,,,,
2018,9,9,8,8,7,9,9
2019,9,9,8,8,8,9,9
2020,9,9,8,8,9,9,9
2021,9,9,9,8,9,9,9
2022,9,9,9,8,9,9,9
2023,9,9,9,8,9,9,9
2024,9,9,9,8,9,9,9


## 7. Create financial ratios for comparison

Raw financial values cannot be directly compared across companies because firms have different sizes.

To make a fair comparison, I convert financial values into ratios relative to revenue.

The following metrics are calculated:

- R&D intensity: Research and Development expense as a percentage of revenue
- Net margin: Net income as a percentage of revenue
- Free cash flow (FCF): Operating cash flow minus capital expenditure
- FCF margin: Free cash flow as a percentage of revenue
- Capex intensity: Capital expenditure as a percentage of revenue

These measures allow comparison of how different sectors invest, generate profits, and fund growth.

In [21]:
panel = panel.rename(columns={
    "ResearchAndDevelopmentExpense": "rd",
    "NetIncomeLoss": "net_income",
    "NetCashProvidedByUsedInOperatingActivities": "ocf",
    "PaymentsToAcquirePropertyPlantAndEquipment": "capex",
    "Revenue": "revenue",
})

panel["rd_intensity"] = panel["rd"] / panel["revenue"]
panel["net_margin"] = panel["net_income"] / panel["revenue"]
panel["fcf"] = panel["ocf"] - panel["capex"]
panel["fcf_margin"] = panel["fcf"] / panel["revenue"]
panel["capex_intensity"] = panel["capex"] / panel["revenue"]

## 8. Check completeness of the final panel

Before starting the analysis, I verify that the dataset contains observations for every company-year combination between 2018 and 2024.

This check identifies any missing financial values in the variables required for the analysis. Missing observations can affect ratio calculations and sector comparisons, so they need to be identified before producing findings.

In [22]:
expected = pd.MultiIndex.from_product(
    [companies.keys(), range(2018, 2025)], names=["ticker", "year"]
)

check = panel.set_index(["ticker", "year"]).reindex(expected)

value_cols = ["revenue", "rd", "net_income", "ocf", "capex"]

missing = check[value_cols].isna().stack()
missing[missing].reset_index().rename(columns={"level_2": "column"})[["ticker", "year", "column"]]

,ticker,year,column
0,NVDA,2018,revenue
1,NVDA,2018,capex
2,NVDA,2019,capex
3,NVDA,2020,capex
4,PFE,2018,revenue
5,PFE,2019,revenue
6,DUK,2018,rd
7,DUK,2019,rd
8,DUK,2020,rd
9,DUK,2021,rd


In [23]:
pd.read_csv("../data/processed/company_financials.csv").shape

(63, 14)

## 9. Prepare and export the final dataset

The financial values are converted from dollars to billions to make the results easier to interpret.

The final dataset combines:
- company information,
- annual financial values,
- calculated financial ratios.

The processed data is then exported as a CSV file and will be used for analysis and visualisation in NB03.

In [24]:
money_cols = ["revenue", "rd", "net_income", "ocf", "capex", "fcf"]

for col in money_cols:
    panel[col] = panel[col] / 1e9

panel = panel.rename(columns={c: f"{c}_bn" for c in money_cols})

panel = panel[[
    "ticker", "name", "sector", "year",
    "revenue_bn", "rd_bn", "net_income_bn", "ocf_bn", "capex_bn", "fcf_bn",
    "rd_intensity", "net_margin", "fcf_margin", "capex_intensity",
]].sort_values(["sector", "ticker", "year"])

panel.to_csv("../data/processed/company_financials.csv", index=False)
panel.head()

,ticker,name,sector,year,revenue_bn,rd_bn,net_income_bn,ocf_bn,capex_bn,fcf_bn,rd_intensity,net_margin,fcf_margin,capex_intensity
30,CVX,Chevron,Energy,2018,166.339,0.453,14.824,30.618,13.792,16.826,0.002723,0.089119,0.101155,0.082915
31,CVX,Chevron,Energy,2019,146.516,0.500,2.924,27.314,14.116,13.198,0.003413,0.019957,0.090079,0.096344
32,CVX,Chevron,Energy,2020,94.692,0.435,-5.543,10.577,8.922,1.655,0.004594,-0.058537,0.017478,0.094221
33,CVX,Chevron,Energy,2021,162.465,0.268,15.600,29.187,8.056,21.131,0.001650,0.096021,0.130065,0.049586
34,CVX,Chevron,Energy,2022,246.252,0.268,35.500,49.602,11.974,37.628,0.001088,0.144161,0.152803,0.048625


In [25]:
panel.shape

(63, 14)